# 🏢 Cetak Denah Ruangan Bikom
Notebook ini digunakan untuk menghasilkan *template* Word (.docx) ruangan siap cetak berdasarkan Jadwal Excel.

In [ ]:
# Install dependencies
!pip install pandas openpyxl python-docx

In [ ]:
import pandas as pd
import docx
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files
import io

### 1. Upload File Excel Jadwal
Silakan jalankan sel di bawah ini dan unggah file Excel (misal: "Salinan dari Jadwal & Denah Bikom 5 Dataset.xlsx").

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"File '{file_name}' berhasil diunggah!")

### 2. Generate DOCX
Pilih nama sheet yang akan diproses dan jalankan kode di bawah ini.

In [ ]:
# Masukkan nama-nama sheet yang ingin diproses (sesuaikan dengan yang ada di file Excel)
target_sheets = ["Jadwal Jumat", "Jadwal Sabtu", "Jadwal Minggu"]

doc = docx.Document()
excel_file = pd.ExcelFile(file_name)

for sheet in target_sheets:
    if sheet not in excel_file.sheet_names:
        print(f"Sheet '{sheet}' tidak ditemukan, dilewati.")
        continue
        
    print(f"Memproses sheet: {sheet}...")
    df = pd.read_excel(file_name, sheet_name=sheet, header=None)
    
    # Cari tanggal dari baris pertama jika ada
    tanggal_hari = "Hari, Tanggal"
    first_val = str(df.iloc[0, 0])
    if " - " in first_val:
        tanggal_hari = first_val.split(" - ")[-1].strip().title()
    else:
        tanggal_hari = sheet.title()
        
    # Cari semua sel yang berisi "Ruangan"
    ruangan_anchors = []
    for r in range(len(df)):
        for c in range(len(df.columns)):
            val = str(df.iloc[r, c]).strip()
            if val.lower() == "ruangan":
                ruangan_anchors.append((r, c))
    
    for r, c in ruangan_anchors:
        try:
            # Ekstrak Data
            dosen = str(df.iloc[r, c+3]) if c+3 < len(df.columns) else "-"
            nama_ruangan = str(df.iloc[r+2, c]) if r+2 < len(df) else "-"
            skema = str(df.iloc[r+2, c+8]) if c+8 < len(df.columns) and r+2 < len(df) else "-"
            
            if pd.isna(nama_ruangan) or nama_ruangan == "nan" or not nama_ruangan:
                continue
                
            # Kumpulkan tim
            teams = []
            row_idx = r + 2
            while row_idx < len(df) and c+3 < len(df.columns):
                ketua = str(df.iloc[row_idx, c+3]).strip()
                if ketua == "nan" or not ketua:
                    break
                teams.append(ketua)
                row_idx += 1
                
            if not teams:
                continue 
                
            # === Tulis ke DOCX ===
            # 1. Ruangan
            p_ruangan = doc.add_paragraph()
            p_ruangan.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run_ruangan = p_ruangan.add_run(nama_ruangan)
            run_ruangan.bold = True
            run_ruangan.font.size = Pt(80)
            run_ruangan.font.name = 'Times New Roman'
            
            # 2. Skema PKM
            p_skema = doc.add_paragraph()
            p_skema.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run_skema = p_skema.add_run(skema)
            run_skema.bold = True
            run_skema.font.size = Pt(60)
            run_skema.font.name = 'Times New Roman'
            
            # 3. Hari dan Tanggal
            p_hari = doc.add_paragraph()
            p_hari.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run_hari = p_hari.add_run(tanggal_hari)
            run_hari.font.size = Pt(34)
            run_hari.font.name = 'Times New Roman'
            
            doc.add_paragraph() 
            
            # 4. Dosen Pembimbing
            p_dosen = doc.add_paragraph()
            p_dosen.alignment = WD_ALIGN_PARAGRAPH.CENTER
            dosen_list = [d.strip() for d in str(dosen).split(";") if d.strip()]
            for i, d in enumerate(dosen_list):
                run_d = p_dosen.add_run(d)
                if i < len(dosen_list) - 1:
                    p_dosen.add_run("\n")
                run_d.font.size = Pt(16)
                run_d.font.name = 'Times New Roman'
            
            doc.add_paragraph()
            
            # 5. Daftar Tim
            p_daftar = doc.add_paragraph()
            p_daftar.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run_daftar = p_daftar.add_run("Daftar Tim")
            run_daftar.bold = True
            run_daftar.font.size = Pt(12)
            run_daftar.font.name = 'Times New Roman'
            
            # Table
            table = doc.add_table(rows=1, cols=1)
            table.style = 'Table Grid'
            hdr_cells = table.rows[0].cells
            hdr_cells[0].text = ''
            p_hdr = hdr_cells[0].paragraphs[0]
            p_hdr.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run_hdr = p_hdr.add_run('Nama Ketua')
            run_hdr.bold = True
            run_hdr.font.size = Pt(12)
            run_hdr.font.name = 'Times New Roman'
            
            for ketua in teams:
                row_cells = table.add_row().cells
                row_cells[0].text = ''
                p_row = row_cells[0].paragraphs[0]
                p_row.alignment = WD_ALIGN_PARAGRAPH.CENTER
                run_row = p_row.add_run(ketua)
                run_row.font.size = Pt(12)
                run_row.font.name = 'Times New Roman'
                
            doc.add_page_break()
            
        except Exception as ex:
            print(f"Error parsing blok at {r}, {c}: {ex}")

output_filename = "Denah_Ruangan_Bikom.docx"
doc.save(output_filename)
print(f"File {output_filename} berhasil dibuat!")

### 3. Download Hasil DOCX
Jalankan sel ini untuk mengunduh file `.docx` ke komputermu.

In [ ]:
files.download(output_filename)